# Opioids Project: Data Cleaning

Ra'Kira Nelson and Alexa Fahrer

In [1]:
import pandas as pd

pd.set_option("mode.copy_on_write", True)

## Prescriptions

In [2]:
prescriptions_raw = pd.read_parquet("data/ids590_opioids_by_drug_county_year.parquet")
prescriptions = prescriptions_raw.copy()

In [3]:
prescriptions["mme_conversion_factor"] = (
    prescriptions["mme_conversion_factor"].to_numpy().astype("float64")
)
prescriptions["calc_base_wt_in_gm"] = (
    prescriptions["calc_base_wt_in_gm"].to_numpy().astype("float64")
)
prescriptions["buyer_county"] = prescriptions["buyer_county"].str.upper().str.strip()
prescriptions["buyer_state"] = prescriptions["buyer_state"].str.upper().str.strip()

prescriptions = prescriptions[
    ~prescriptions["buyer_state"].isin(["AK", "PR", "VI", "GU", "MP", "AS", "PW"])
]
prescriptions

,buyer_state,buyer_county,year,drug_name,mme_conversion_factor,calc_base_wt_in_gm
2682,AL,AUTAUGA,2006,BUPRENORPHINE,30.00,7.980816
2683,AL,AUTAUGA,2006,BUPRENORPHINE,75.00,0.019533
2684,AL,AUTAUGA,2006,CODEINE,0.15,2217.649100
2685,AL,AUTAUGA,2006,DIHYDROCODEINE,0.25,59.102157
2686,AL,AUTAUGA,2006,FENTANYL,100.00,225.711500
...,...,...,...,...,...,...
563626,WY,WESTON,2019,METHADONE,4.00,25.490400
563627,WY,WESTON,2019,MORPHINE,1.00,332.579520
563628,WY,WESTON,2019,OXYCODONE,1.50,621.053064
563629,WY,WESTON,2019,OXYMORPHONE,3.00,9.634680


In [4]:
prescriptions["mme"] = (
    prescriptions["calc_base_wt_in_gm"] * 1000 * prescriptions["mme_conversion_factor"]
)
prescriptions_condensed = prescriptions.groupby(
    ["buyer_state", "buyer_county", "year"], as_index=False
)["mme"].sum()
prescriptions_condensed

,buyer_state,buyer_county,year,mme
0,AL,AUTAUGA,2006,5.268607e+07
1,AL,AUTAUGA,2007,5.458566e+07
2,AL,AUTAUGA,2008,5.777668e+07
3,AL,AUTAUGA,2009,5.953256e+07
4,AL,AUTAUGA,2010,6.410113e+07
...,...,...,...,...
41850,WY,WESTON,2015,4.318172e+06
41851,WY,WESTON,2016,3.396718e+06
41852,WY,WESTON,2017,3.098053e+06
41853,WY,WESTON,2018,2.766198e+06


## FIPS

In [5]:
fips = pd.read_excel("data/US_FIPS_Codes.xls", skiprows=1)

In [6]:
fips["fips"] = fips["FIPS State"].astype(str).str.zfill(2) + fips["FIPS County"].astype(
    str
).str.zfill(3)

fips = fips[~fips["State"].isin(["AK"])]

us_state_abbrev = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "District of Columbia": "DC",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
}

fips["state_abbrev"] = fips["State"].map(us_state_abbrev)
fips["County Name"] = fips["County Name"].str.upper().str.strip()
fips["state_abbrev"] = fips["state_abbrev"].str.upper().str.strip()

fips["County Name"] = (
    fips["County Name"]
    .str.upper()
    .str.strip()
    .str.replace(r"^ST[.\s]+", "SAINT ", regex=True)
)

fips

,State,County Name,FIPS State,FIPS County,fips,state_abbrev
0,Alabama,AUTAUGA,1,1,01001,AL
1,Alabama,BALDWIN,1,3,01003,AL
2,Alabama,BARBOUR,1,5,01005,AL
3,Alabama,BIBB,1,7,01007,AL
4,Alabama,BLOUNT,1,9,01009,AL
...,...,...,...,...,...,...
3137,Wyoming,SWEETWATER,56,37,56037,WY
3138,Wyoming,TETON,56,39,56039,WY
3139,Wyoming,UINTA,56,41,56041,WY
3140,Wyoming,WASHAKIE,56,43,56043,WY


In [7]:
prescriptions_fips_merged = pd.merge(
    prescriptions_condensed,
    fips[["state_abbrev", "County Name", "fips"]],
    left_on=["buyer_state", "buyer_county"],
    right_on=["state_abbrev", "County Name"],
    how="left",
    indicator=True,
    validate="m:1",
)

prescriptions_merged = prescriptions_fips_merged.drop(
    columns=["state_abbrev", "County Name", "_merge"]
).copy()

In [8]:
print(prescriptions_fips_merged["_merge"].value_counts())

_merge
both          41729
left_only       126
right_only        0
Name: count, dtype: int64


In [9]:
unmatched = prescriptions_fips_merged[
    prescriptions_fips_merged["_merge"] == "left_only"
]
unmatched[["buyer_state", "buyer_county"]].drop_duplicates().sort_values(
    ["buyer_state", "buyer_county"]
)

,buyer_state,buyer_county
1606,AR,MONTGOMERY
5467,GA,DEKALB
9260,IL,DEKALB
9302,IL,DUPAGE
11443,IN,ST JOSEPH
15498,LA,ST JOHN THE BAPTIST
20126,MO,SAINTE GENEVIEVE
20603,MS,DESOTO
38766,VA,SALEM


In [10]:
prescriptions_merged

,buyer_state,buyer_county,year,mme,fips
0,AL,AUTAUGA,2006,5.268607e+07,01001
1,AL,AUTAUGA,2007,5.458566e+07,01001
2,AL,AUTAUGA,2008,5.777668e+07,01001
3,AL,AUTAUGA,2009,5.953256e+07,01001
4,AL,AUTAUGA,2010,6.410113e+07,01001
...,...,...,...,...,...
41850,WY,WESTON,2015,4.318172e+06,56045
41851,WY,WESTON,2016,3.396718e+06,56045
41852,WY,WESTON,2017,3.098053e+06,56045
41853,WY,WESTON,2018,2.766198e+06,56045


## Deaths

In [11]:
deaths_dfs = {}
for year in range(2003, 2016):
    key = f"deaths_{year}"
    url = (
        "https://media.githubusercontent.com/media/nickeubank/ids540_opioid_data/"
        f"refs/heads/main/vitalstatistics/Underlying%20Cause%20of%20Death%2C%20{year}.txt"
    )

    df = pd.read_csv(url, sep="\t", skipfooter=15, engine="python")
    df = df.drop(columns=["Notes"])
    deaths_dfs[key] = df

deaths = pd.concat(
    [deaths_dfs[f"deaths_{year}"].assign(year=year) for year in range(2006, 2016)],
    ignore_index=True,
)

In [12]:
deaths["Year"] = pd.to_numeric(deaths["Year"], errors="coerce").astype("Int64")
deaths["Deaths"] = pd.to_numeric(deaths["Deaths"], errors="coerce").astype("Int64")
deaths = deaths[
    deaths["Drug/Alcohol Induced Cause"]
    == "Drug poisonings (overdose) Unintentional (X40-X44)"
]
deaths["fips"] = deaths["County Code"].astype(str).str.zfill(5)
deaths = deaths.drop(columns=["Year Code", "year", "County Code"])
deaths = deaths[~deaths["County"].str.endswith(", AK", na=False)]
deaths = deaths.rename(columns={"Year": "year"})
deaths = deaths.rename(columns={"Deaths": "deaths"})
deaths

,County,year,Drug/Alcohol Induced Cause,Drug/Alcohol Induced Cause Code,deaths,fips
1,"Baldwin County, AL",2006,Drug poisonings (overdose) Unintentional (X40-...,D1,11,01003
12,"Chilton County, AL",2006,Drug poisonings (overdose) Unintentional (X40-...,D1,13,01021
39,"Jefferson County, AL",2006,Drug poisonings (overdose) Unintentional (X40-...,D1,55,01073
55,"Mobile County, AL",2006,Drug poisonings (overdose) Unintentional (X40-...,D1,23,01097
60,"Montgomery County, AL",2006,Drug poisonings (overdose) Unintentional (X40-...,D1,12,01101
...,...,...,...,...,...,...
44778,"Waukesha County, WI",2015,Drug poisonings (overdose) Unintentional (X40-...,D1,34,55133
44784,"Winnebago County, WI",2015,Drug poisonings (overdose) Unintentional (X40-...,D1,22,55139
44794,"Fremont County, WY",2015,Drug poisonings (overdose) Unintentional (X40-...,D1,10,56013
44800,"Laramie County, WY",2015,Drug poisonings (overdose) Unintentional (X40-...,D1,13,56021


## Population

In [13]:
pop_1 = pd.read_csv("data/co-est00int-tot.csv", encoding="latin1")
pop_2 = pd.read_csv("data/co-est2020.csv", encoding="latin1")

In [14]:
p1_sub = pop_1[
    [
        "STATE",
        "COUNTY",
        "STNAME",
        "CTYNAME",
        "POPESTIMATE2006",
        "POPESTIMATE2007",
        "POPESTIMATE2008",
        "POPESTIMATE2009",
    ]
].copy()
p2_sub = pop_2[
    [
        "STATE",
        "COUNTY",
        "STNAME",
        "CTYNAME",
        "POPESTIMATE2010",
        "POPESTIMATE2011",
        "POPESTIMATE2012",
        "POPESTIMATE2013",
        "POPESTIMATE2014",
        "POPESTIMATE2015",
    ]
].copy()
pop_merged = p1_sub.merge(
    p2_sub, on=["STATE", "COUNTY", "STNAME", "CTYNAME"], how="inner"
)
pop_merged["fips"] = pop_merged["STATE"].astype(str).str.zfill(2) + pop_merged[
    "COUNTY"
].astype(str).str.zfill(3)
pop_merged = pop_merged[~pop_merged["STNAME"].isin(["Alaska"])]

In [15]:
pop_merged

,STATE,COUNTY,STNAME,CTYNAME,POPESTIMATE2006,POPESTIMATE2007,POPESTIMATE2008,POPESTIMATE2009,POPESTIMATE2010,POPESTIMATE2011,POPESTIMATE2012,POPESTIMATE2013,POPESTIMATE2014,POPESTIMATE2015,fips
0,1,0,Alabama,Alabama,4628981,4672840,4718206,4757938,4785514,4799642,4816632,4831586,4843737,4854803,01000
1,1,1,Alabama,Autauga County,51328,52405,53277,54135,54761,55229,54970,54747,54922,54903,01001
2,1,3,Alabama,Baldwin County,168121,172404,175827,179406,183121,186579,190203,194978,199306,203101,01003
3,1,5,Alabama,Barbour County,27861,27757,27808,27657,27325,27344,27172,26946,26768,26300,01005
4,1,7,Alabama,Bibb County,22099,22438,22705,22941,22858,22736,22657,22510,22541,22553,01007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3183,56,37,Wyoming,Sweetwater County,39749,41470,42358,44133,43580,44000,45032,45189,44996,44780,56037
3184,56,39,Wyoming,Teton County,20014,20472,20988,21232,21298,21422,21643,22335,22801,23083,56039
3185,56,41,Wyoming,Uinta County,19709,20171,20613,21054,21090,20901,21008,20969,20835,20777,56041
3186,56,43,Wyoming,Washakie County,7979,8169,8229,8423,8531,8451,8410,8417,8277,8282,56043


In [16]:
year_cols = [c for c in pop_merged.columns if c.startswith("POPESTIMATE")]
pop_long = pop_merged.melt(
    id_vars=["fips", "STNAME", "CTYNAME"],
    value_vars=year_cols,
    var_name="pop_var",
    value_name="population",
)
pop_long["Year"] = pop_long["pop_var"].str.extract(r"(\d{4})").astype(int)
pop_long = pop_long.drop(columns=["pop_var"])
pop_long = pop_long.rename(columns={"STNAME": "state"})
pop_long = pop_long.rename(columns={"CTYNAME": "county"})
pop_long = pop_long.rename(columns={"Year": "year"})
pop_long["state"] = pop_long["state"].str.upper().str.strip()
pop_long["county"] = pop_long["county"].str.upper().str.strip()
pop_long

,fips,state,county,population,year
0,01000,ALABAMA,ALABAMA,4628981,2006
1,01001,ALABAMA,AUTAUGA COUNTY,51328,2006
2,01003,ALABAMA,BALDWIN COUNTY,168121,2006
3,01005,ALABAMA,BARBOUR COUNTY,27861,2006
4,01007,ALABAMA,BIBB COUNTY,22099,2006
...,...,...,...,...,...
31605,56037,WYOMING,SWEETWATER COUNTY,44780,2015
31606,56039,WYOMING,TETON COUNTY,23083,2015
31607,56041,WYOMING,UINTA COUNTY,20777,2015
31608,56043,WYOMING,WASHAKIE COUNTY,8282,2015


## Merging

In [17]:
deaths

,County,year,Drug/Alcohol Induced Cause,Drug/Alcohol Induced Cause Code,deaths,fips
1,"Baldwin County, AL",2006,Drug poisonings (overdose) Unintentional (X40-...,D1,11,01003
12,"Chilton County, AL",2006,Drug poisonings (overdose) Unintentional (X40-...,D1,13,01021
39,"Jefferson County, AL",2006,Drug poisonings (overdose) Unintentional (X40-...,D1,55,01073
55,"Mobile County, AL",2006,Drug poisonings (overdose) Unintentional (X40-...,D1,23,01097
60,"Montgomery County, AL",2006,Drug poisonings (overdose) Unintentional (X40-...,D1,12,01101
...,...,...,...,...,...,...
44778,"Waukesha County, WI",2015,Drug poisonings (overdose) Unintentional (X40-...,D1,34,55133
44784,"Winnebago County, WI",2015,Drug poisonings (overdose) Unintentional (X40-...,D1,22,55139
44794,"Fremont County, WY",2015,Drug poisonings (overdose) Unintentional (X40-...,D1,10,56013
44800,"Laramie County, WY",2015,Drug poisonings (overdose) Unintentional (X40-...,D1,13,56021


In [18]:
pop_long

,fips,state,county,population,year
0,01000,ALABAMA,ALABAMA,4628981,2006
1,01001,ALABAMA,AUTAUGA COUNTY,51328,2006
2,01003,ALABAMA,BALDWIN COUNTY,168121,2006
3,01005,ALABAMA,BARBOUR COUNTY,27861,2006
4,01007,ALABAMA,BIBB COUNTY,22099,2006
...,...,...,...,...,...
31605,56037,WYOMING,SWEETWATER COUNTY,44780,2015
31606,56039,WYOMING,TETON COUNTY,23083,2015
31607,56041,WYOMING,UINTA COUNTY,20777,2015
31608,56043,WYOMING,WASHAKIE COUNTY,8282,2015


In [19]:
prescriptions_merged

,buyer_state,buyer_county,year,mme,fips
0,AL,AUTAUGA,2006,5.268607e+07,01001
1,AL,AUTAUGA,2007,5.458566e+07,01001
2,AL,AUTAUGA,2008,5.777668e+07,01001
3,AL,AUTAUGA,2009,5.953256e+07,01001
4,AL,AUTAUGA,2010,6.410113e+07,01001
...,...,...,...,...,...
41850,WY,WESTON,2015,4.318172e+06,56045
41851,WY,WESTON,2016,3.396718e+06,56045
41852,WY,WESTON,2017,3.098053e+06,56045
41853,WY,WESTON,2018,2.766198e+06,56045


In [20]:
presc_deaths = pd.merge(
    prescriptions_merged,
    deaths[["fips", "County", "year", "deaths"]],
    on=["fips", "year"],
    how="outer",
)

opioids = pd.merge(
    presc_deaths,
    pop_long[["fips", "year", "county", "state", "population"]],
    on=["fips", "year"],
    how="left",
)

opioids = opioids[
    [
        "fips",
        "buyer_county",
        "County",
        "county",
        "state",
        "buyer_state",
        "year",
        "mme",
        "deaths",
        "population",
    ]
]

print(opioids.shape)
opioids.sample(20)

(41897, 10)


,fips,buyer_county,County,county,state,buyer_state,year,mme,deaths,population
26361,37179,UNION,NaN,NaN,NaN,NC,2019,1.936192e+08,<NA>,NaN
21726,30043,JEFFERSON,NaN,NaN,NaN,MT,2016,2.854256e+06,<NA>,NaN
12223,20083,HODGEMAN,NaN,HODGEMAN COUNTY,KANSAS,KS,2010,1.446913e+06,<NA>,1920.0
21912,30075,POWDER RIVER,NaN,POWDER RIVER COUNTY,MONTANA,MT,2006,9.748415e+05,<NA>,1794.0
15730,23015,LINCOLN,NaN,LINCOLN COUNTY,MAINE,ME,2010,4.408034e+07,<NA>,34384.0
33813,48029,BEXAR,"Bexar County, TX",BEXAR COUNTY,TEXAS,TX,2011,1.009229e+09,227,1755360.0
29151,40111,OKMULGEE,NaN,OKMULGEE COUNTY,OKLAHOMA,OK,2013,3.611523e+07,<NA>,39404.0
2092,05137,STONE,NaN,NaN,NaN,AR,2016,1.334643e+07,<NA>,NaN
12859,20173,SEDGWICK,NaN,NaN,NaN,KS,2016,4.947015e+08,<NA>,NaN
25946,37121,MITCHELL,NaN,MITCHELL COUNTY,NORTH CAROLINA,NC,2010,2.557120e+07,<NA>,15511.0


## Preparation

In [21]:
opioids_clean = opioids.copy()
opioids_clean.shape

(41897, 10)

In [22]:
opioids_clean.sample(20)

,fips,buyer_county,County,county,state,buyer_state,year,mme,deaths,population
26111,37145,PERSON,NaN,PERSON COUNTY,NORTH CAROLINA,NC,2007,3.614919e+07,<NA>,38656.0
6958,13317,WILKES,NaN,WILKES COUNTY,GEORGIA,GA,2010,6.023255e+06,<NA>,10389.0
6881,13305,WAYNE,NaN,WAYNE COUNTY,GEORGIA,GA,2009,3.362633e+07,<NA>,30016.0
3519,08083,MONTEZUMA,NaN,MONTEZUMA COUNTY,COLORADO,CO,2015,2.525972e+07,<NA>,25763.0
32742,47063,HAMBLEN,"Hamblen County, TN",HAMBLEN COUNTY,TENNESSEE,TN,2010,2.492966e+08,13,62509.0
37858,51069,FREDERICK,NaN,FREDERICK COUNTY,VIRGINIA,VA,2012,4.459546e+07,<NA>,80247.0
3155,08025,CROWLEY,NaN,CROWLEY COUNTY,COLORADO,CO,2015,4.215765e+06,<NA>,5633.0
6032,13175,LAURENS,NaN,LAURENS COUNTY,GEORGIA,GA,2015,9.946381e+07,<NA>,47491.0
10705,19061,DUBUQUE,NaN,DUBUQUE COUNTY,IOWA,IA,2007,3.244850e+07,<NA>,92130.0
4707,12109,SAINT JOHNS,NaN,NaN,NaN,FL,2016,2.159053e+08,<NA>,NaN


In [23]:
opioids_clean["deaths"] = pd.to_numeric(
    opioids_clean["deaths"], errors="coerce"
).astype("Int64")
opioids_clean["population"] = opioids_clean["population"].astype("Int64")

In [24]:
opioids_clean["policy_state"] = (opioids_clean["state"] == "FLORIDA").astype("boolean")
opioids_clean["post"] = (opioids_clean["year"] >= 2010).astype(int)
opioids_clean["relative_year"] = opioids_clean["year"] - 2010
opioids_clean["mme_per_1000"] = (
    opioids_clean["mme"] / opioids_clean["population"] * 1000
)
opioids_clean["overdose_per_100k"] = (
    opioids_clean["deaths"] / opioids_clean["population"] * 100000
)
opioids_clean["relative_year"] = opioids_clean["year"] - 2010

In [25]:
opioids_clean.dtypes

fips                  object
buyer_county          object
County                object
county                object
state                 object
buyer_state           object
year                   Int64
mme                  float64
deaths                 Int64
population             Int64
policy_state         boolean
post                   int64
relative_year          Int64
mme_per_1000         Float64
overdose_per_100k    Float64
dtype: object

In [26]:
opioids_deaths = opioids_clean.copy()
avg_pop = opioids_deaths.groupby("fips")["population"].mean()
big_fips = avg_pop[avg_pop >= 350000].index
opioids_deaths = opioids_deaths[opioids_deaths["fips"].isin(big_fips)]

In [27]:
opioids_clean.to_csv("data/opioids_clean.csv")
opioids_deaths.to_csv("data/opioids_deaths.csv")